# Replication notebook: paper figure 3 (sample signals)

Reproduces the four panels of Figure 3 — one channel each of:

* **A.** iEEG intra-subject
* **B.** iEEG cross-subject
* **C.** EEG intra-subject
* **D.** EEG cross-subject

each plotted with the original vs reconstructed mean-trial trace and
horizontal/vertical scale bars.

The original notebook (`visualize_cosyn.ipynb`) loaded hand-picked
`.npy` files from a fixed local path. This version locates the
equivalent `Mean_Channel_*.npz` files produced by
`fixed/visualize.py::save_mse_all_trials`, so re-running the pipeline
and re-running this notebook is enough to refresh the figure.

## Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# Make the fixed/ package and the notebooks helper importable.
_NB_DIR = Path.cwd()
_PKG = _NB_DIR.parent if _NB_DIR.name == 'notebooks' else _NB_DIR
for p in (str(_PKG), str(_PKG / 'notebooks')):
    if p not in sys.path:
        sys.path.insert(0, p)

from _helpers import load_paths, load_mean_channel

paths = load_paths()
RESULTS_DIR = paths.results_dir

FIG_OUT = _NB_DIR / 'figures'
FIG_OUT.mkdir(exist_ok=True)

NEIGHBORHOOD_METHOD = 'atlas'
MODEL_TYPE = 'instantaneous'

## Choose the four panel examples

Pick (subject, channel, mask_intensity) per panel. For the intra-subject
panels we use `mask_intensity=1.0` (full local mask, matching the
manuscript). For the cross-subject panels we point at the per-electrode
*best* signal written by `Test_new_subject_iEEG.py` (`Best_Channel_*.npy`)
for iEEG, and at the `Mean_Channel_*.npz` of the `test_new` run for EEG.

Edit these to match whichever channels you want to display.

In [ ]:
# (dataset_type, mode, subject, channel, mask_intensity, panel title)
PANELS = [
    ('iEEG', 'train',    0, 30, 1.0, 'A. iEEG intra-subject'),
    ('iEEG', 'test_new', 0, 35, 1.0, 'B. iEEG cross-subject'),
    ('EEG',  'train',    0,  1, 1.0, 'C. EEG intra-subject'),
    ('EEG',  'test_new', 0,  2, 1.0, 'D. EEG cross-subject'),
]

# Scale-bar configuration (matches the original cosyn figure).
TIME_SCALE_MS = 500
AMPLITUDE_SCALE = 0.2

## Helpers

In [ ]:
def _load_panel(dataset_type, mode, subject, channel, mask_intensity):
    """Locate the data for one panel.

    * For iEEG cross-subject runs, prefer the `Best_Channel_<channel>.npy`
      written by `Test_new_subject_iEEG.py`, which contains the
      best-donor signal selected from the validation set.
    * Otherwise fall back to the standard `Mean_Channel_<channel+1>.npz`
      written by `save_mse_all_trials`.
    """
    if dataset_type == 'iEEG' and mode == 'test_new':
        best = (Path(RESULTS_DIR) / 'plot' / f'sub_{subject}'
                / f'Best_Channel_{channel}.npy')
        if best.exists():
            arr = np.load(best, allow_pickle=True)
            # The Test_new_subject_iEEG.py script saves
            # [orig, recon, time_axis] for each best channel.
            orig, recon, time_axis = arr
            best_Dcorr = np.load(best.parent / 'best_Dcorr.npy')
            Dcorr = float(best_Dcorr[channel]) if channel < len(best_Dcorr) else float('nan')
            return dict(time_axis=np.asarray(time_axis, dtype=float),
                         orig=np.asarray(orig, dtype=float),
                         recon=np.asarray(recon, dtype=float),
                         Dcorr=Dcorr,
                         source_file=str(best))

    return load_mean_channel(
        RESULTS_DIR, dataset_type, mode, subject, mask_intensity, channel,
        NEIGHBORHOOD_METHOD, MODEL_TYPE,
    )

In [ ]:
def _draw_scale_bars(ax, time_scale_ms=TIME_SCALE_MS, amp_scale=AMPLITUDE_SCALE):
    """Add an L-shaped time/amplitude scale bar to the bottom-left."""
    xmin, xmax = ax.get_xlim()
    ymin, ymax = ax.get_ylim()
    y_offset = (ymax - ymin) * 0.1
    x_offset = (xmax - xmin) * 0.05

    # Time bar.
    t0 = xmin + x_offset
    t1 = t0 + time_scale_ms / 1000.0
    ty = ymin - y_offset
    ax.plot([t0, t1], [ty, ty], color='black', linewidth=3, solid_capstyle='butt')
    ax.text(t0 + time_scale_ms / 2000.0, ty - y_offset * 0.5,
             f'{time_scale_ms} ms',
             ha='center', va='top', fontsize=24, fontweight='bold')

    # Amplitude bar.
    ax.plot([t0, t0], [ty, ty + amp_scale],
             color='black', linewidth=3, solid_capstyle='butt')
    label = (f'{amp_scale:.1f} V' if amp_scale >= 1.0
              else f'{amp_scale * 1000:.0f} mV')
    ax.text(t0 - x_offset * 0.5, ty + amp_scale / 2,
             label, ha='center', va='center', rotation=90,
             fontsize=24, fontweight='bold')


def plot_panel(panel_spec):
    dataset_type, mode, subject, channel, mask_intensity, title = panel_spec
    try:
        d = _load_panel(dataset_type, mode, subject, channel, mask_intensity)
    except FileNotFoundError as e:
        print(f'[skip] {title} -- {e}')
        return None

    fig, ax = plt.subplots(figsize=(15, 5), dpi=300)
    ax.plot(d['time_axis'], d['orig'],  alpha=0.9, color='blue')
    ax.plot(d['time_axis'], d['recon'],
             label=f"DistCorr: {d['Dcorr']:.3f}", alpha=0.9, color='red')

    ax.grid(False)
    ax.set_xlabel(''); ax.set_ylabel('')
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

    _draw_scale_bars(ax)
    ax.legend(fontsize=18, loc='upper right')
    plt.tight_layout()

    base = f'fig3_{dataset_type}_{mode}_sub{subject}_ch{channel}'
    fig.savefig(FIG_OUT / f'{base}.png')
    fig.savefig(FIG_OUT / f'{base}.svg')
    plt.show()
    print(f'[ok]  {title}: source = {d["source_file"]}')
    return d['Dcorr']

## Draw the four panels

Each call prints the file used and saves the panel as both PNG and SVG
into `figures/`.

In [ ]:
for spec in PANELS:
    plot_panel(spec)

## Picking different channels

If a panel has a low DistCorr or the trace doesn't look representative,
edit the `PANELS` tuple above. To browse the available channels for a
given (dataset, mode, subject), list the `compare_channel/` directory of
the corresponding run:

```python
from _helpers import find_mean_channel_file
fname = find_mean_channel_file(RESULTS_DIR, 'EEG', 'train', subject=0,
                                mask_intensity=1.0, channel=0)
print(fname.parent)   # list all Mean_Channel_*.npz files here
```

For iEEG cross-subject, the analogous source is
`<results_dir>/plot/sub_<subject>/`, with files named
`Best_Channel_<channel>.npy` (one per electrode).